# 05 — Generation-based cloud fill (TerraMind large) + full pipeline rerun

## Motivation

Clouds weren't a real problem until the biweekly panel arrived. The original 26-site
data was one *before* and one *after* image per site, each a median composite over a
~4.5-month window — long enough that almost every pixel had cloud-free observations
(valid fractions ≈ 1.0). The chip-mean fill was touching almost nothing, so a generative
filling stage would have added complexity for no benefit. The problem became first-order
with the biweekly panel: **14-day windows leave S2 chips only 57–69% valid, so the
chip-mean fill fabricates a third of every image the encoder sees** — flat, texture-free
patches far outside the training distribution of TerraMind's tokenizers (whose TerraMesh
pretraining data was cloud-filtered), and a moving cloud mask that injects
period-to-period noise unrelated to the ground.

This notebook replaces the chip-mean fill with **TerraMind any-to-any generation**
(Jakubik et al., arXiv:2504.11171; github.com/IBM/terramind), then reruns the entire
pipeline of notebooks 01–04 through the reusable modules `panel_lib.py` /
`panel_scm.py` / `panel_ascm.py` and compares against the chip-mean run's results,
which stay untouched on disk.

## Design (Jun's decisions, 2026-08-20)

- **Model** `terramind_v1_large_generate`, used EXACTLY as the IBM repo's generation
  notebooks: `FULL_MODEL_REGISTRY.build(..., modalities=[inp], output_modalities=[out],
  pretrained=True, standardize=True)`, called on raw physical-unit tensors
  `[B, C, 224, 224]`, `timesteps=10`; the output is a **full generated image** in
  physical units (the paper has no inpainting or compositing mode — full replacement).
- **Arm A (the experiment, S1→S2):** every S2 image with any invalid pixels is replaced
  by a chip generated from **its own same-period S1** (radar sees through clouds).
  Fill scope = ALL S2 images, donors + treated, all 20 periods — including the treated
  P09/P10 **before encoding only**, so the latent-space validation compares
  like-with-like. No leakage: conditioning is same-site, same-period only.
- **Arm B (S2→S1):** S1 images with *no acquisition at all* are generated from their
  own same-period observed S2 (the paper's symmetric optical→radar direction),
  recovering fit periods the chip-mean run had to drop. No treated P09/P10 is ever
  generated (all exist — asserted).
- **Ground truth is never generated:** feature- and image-space validation always
  scores against the raw observed chips (nan-aware features; valid pixels only).
- Approved adaptations to the IBM recipe (forced by our data): inputs bilinearly
  resized 101→224; arm-B S2 input carries our 6 reflectance bands in the 12-band slots
  with the rest at the pretraining mean (the tokenizer arm's own convention); Python
  `random` + torch seeded before every call for reproducibility.

## What this can and cannot fix

A cloud fix raises the S2 *input* quality (in-distribution latents, no mask-movement
noise, a floor no longer dominated by fabricated pixels). It does **not** by itself
rescue latent-space SCM: the quality-40 sensitivity arm already showed that clean images
alone leave validation RMSE at ≈ 1 SD. The question this notebook answers is how much of
the S2 error budget the clouds were responsible for.

**Pipeline:** parity gate (modules ≡ notebooks 02/03) → generation → encode → kNN →
SCM → ASCM → decode-vs-truth → effects → comparison. Chip-mean results
(`panel_*.csv`) are read for comparison only.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
import panel_lib as pl

GPU = pl.pick_gpu()   # before any torch import
print("CUDA_VISIBLE_DEVICES =", GPU)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import panel_scm
import panel_ascm

pl.set_plot_style()
TS, LATD = pl.TS, pl.LATD
SENSORS = pl.SENSORS

roster = pl.load_roster()
idx = pl.build_panel_index(roster)                      # 2,400 rows, 2,322 files
lat_orig = pl.load_latents(LATD / "latents_biweekly.npz")
assert len(lat_orig) == 2322
panel_orig = pl.Panel(lat_orig, idx, roster)
print(f"original panel: {len(panel_orig.flat)} latents, D = {panel_orig.D}")

## Parity gate — the modules must reproduce the executed notebooks 02/03

`run_scm` / `run_ascm` on the ORIGINAL chip-mean latents, both arms, compared against
every CSV notebook 02/03 wrote (numeric ≤ 1e-8, strings exact). Nothing new runs unless
this passes.

In [ ]:
# latent_frac_outside_01 is a 0/1-threshold-crossing count: the ASCM weights match
# the executed run only to ~1e-13 (threaded-BLAS summation order in np.linalg.solve),
# which can flip a handful of the 980 borderline latent values across the boundary.
COL_ATOL = {"latent_frac_outside_01": 0.011}

def cmp(name, new, ref, keys, num_atol=1e-8):
    new = new.reset_index(drop=True); ref = ref.reset_index(drop=True)
    assert list(new.columns) == list(ref.columns), (name, list(new.columns))
    assert len(new) == len(ref), (name, len(new), len(ref))
    new = new.sort_values(keys).reset_index(drop=True)
    ref = ref.sort_values(keys).reset_index(drop=True)
    worst = 0.0
    for c in new.columns:
        if pd.api.types.is_numeric_dtype(ref[c]) and not pd.api.types.is_bool_dtype(ref[c]):
            a, b = new[c].to_numpy(dtype=float), ref[c].to_numpy(dtype=float)
            assert (np.isfinite(a) == np.isfinite(b)).all(), (name, c, "nan pattern")
            d = np.abs(a[np.isfinite(a)] - b[np.isfinite(b)])
            w = float(d.max()) if len(d) else 0.0
            worst = max(worst, w)
            assert w <= COL_ATOL.get(c, num_atol), (name, c, w)
        else:
            assert (new[c].astype(str) == ref[c].astype(str)).all(), (name, c)
    print(f"PARITY OK  {name:26s} rows={len(new)}  max|diff|={worst:.2e}")

ARMS = ("main", "quality40")
don0, donors0, _ = pl.knn_donors(panel_orig, arms=ARMS)
cmp("panel_knn_donors", don0, pd.read_csv(TS / "panel_knn_donors.csv"),
    ["arm", "sensor", "treatment_site_id", "rank"])
scm0 = panel_scm.run_scm(panel_orig, donors0, arms=ARMS)
cmp("panel_scm_weights", scm0["weights"], pd.read_csv(TS / "panel_scm_weights.csv"),
    ["arm", "sensor", "treatment_site_id", "fit_window", "donor"])
cmp("panel_scm_validation", scm0["validation"],
    pd.read_csv(TS / "panel_scm_validation.csv"),
    ["arm", "sensor", "treatment_site_id", "scheme", "eval_period"])
cmp("panel_scm_effects", scm0["effects"], pd.read_csv(TS / "panel_scm_effects.csv"),
    ["sensor", "treatment_site_id", "period"])
ascm0 = panel_ascm.run_ascm(panel_orig, donors0, scm0, arms=ARMS)
cmp("panel_ascm_weights", ascm0["weights"],
    pd.read_csv(TS / "panel_ascm_weights.csv"),
    ["arm", "sensor", "treatment_site_id", "fit_window", "donor"])
cmp("panel_ascm_validation", ascm0["validation"],
    pd.read_csv(TS / "panel_ascm_validation.csv"),
    ["arm", "sensor", "treatment_site_id", "scheme", "eval_period"])
cmp("panel_scm_vs_ascm", ascm0["vs_scm"], pd.read_csv(TS / "panel_scm_vs_ascm.csv"),
    ["arm", "sensor", "treatment_site_id", "scheme", "eval_period"])
cmp("panel_ascm_effects", ascm0["effects"],
    pd.read_csv(TS / "panel_ascm_effects.csv"),
    ["sensor", "treatment_site_id", "period"])
print("\nALL PARITY CHECKS PASSED — the modules are the notebooks 02/03 pipeline.")

## Generation plan + provenance

- **arm A (S1→S2):** S2 chip on disk with `valid_pixel_fraction < 1` and same-period S1
  on disk → `generated_s1_to_s2` (full replacement). Same-period S1 missing →
  `chipmean_fallback` (original path, recorded). Fully valid → `none`.
- **arm B (S2→S1):** S1 with no acquisition, same-period S2 on disk →
  `generated_s2_to_s1` (whole image). Neither sensor on disk → stays missing.

In [ ]:
exists = idx.set_index(["site_id", "sensor", "period_id"])["file_exists"].to_dict()
prov_rows = []
for r in idx.itertuples():
    other = "sentinel1" if r.sensor == "sentinel2" else "sentinel2"
    partner = exists.get((r.site_id, other, r.period_id), False)
    if r.sensor == "sentinel2" and r.file_exists:
        if r.valid_pixel_fraction >= 1.0:
            fm = "none"
        elif partner:
            fm = "generated_s1_to_s2"
        else:
            fm = "chipmean_fallback"
        arm = "A" if fm == "generated_s1_to_s2" else ""
    elif r.sensor == "sentinel1" and not r.file_exists and partner:
        fm, arm = "generated_s2_to_s1", "B"
    elif r.file_exists:
        fm, arm = "none", ""
    else:
        fm, arm = "still_missing", ""
    prov_rows.append({"site_id": r.site_id, "sensor": r.sensor,
                      "period_id": r.period_id, "seq": r.seq,
                      "valid_pixel_fraction": r.valid_pixel_fraction,
                      "fill_method": fm, "arm": arm})
prov = pd.DataFrame(prov_rows)
assert len(prov) == 2400
prov.to_csv(TS / "panel_genfill_provenance.csv", index=False)
print("saved:", TS / "panel_genfill_provenance.csv")
print(prov.groupby(["sensor", "fill_method"]).size())

# no treated P09/P10 image is ever generated (fill BEFORE encoding is allowed for S2 by
# the fill-scope decision, but arm B must never fabricate a validation-period S1)
tb = prov[(prov["sensor"] == "sentinel1")
          & (prov["fill_method"] == "generated_s2_to_s1")
          & prov["site_id"].str.startswith("treatment")
          & prov["seq"].isin([9, 10])]
assert len(tb) == 0, tb
genA = prov.query("fill_method == 'generated_s1_to_s2'")
genB = prov.query("fill_method == 'generated_s2_to_s1'")
print(f"\narm A: {len(genA)} S2 chips to generate | "
      f"arm B: {len(genB)} whole S1 images to generate")
print("arm B by site half:", genB.groupby(genB["seq"] <= 10).size().to_dict(),
      "(True = pre-hurricane)")

## Arm A — generate S2 from same-period S1 (full replacement)

IBM recipe verbatim: raw S1 (VV,VH in dB) at 224², `model(x, timesteps=10)`, output
12-band S2 in DN units at 224². The 6 project bands are extracted via the same
IDX12 mapping the tokenizer path uses. Cache: float16 (rounded once so a cache reload
encodes identically), load-if-exists. Determinism gate: first batch regenerated.

In [ ]:
import torch
import torch.nn.functional as F

tokb = pl.load_tokenizers()

# raw-unit input preps for the generator (standardize=True handles normalization)
def gen_input_s1(site, period_id):
    row = idx.query("site_id == @site and sensor == 'sentinel1' and "
                    "period_id == @period_id").iloc[0]
    chip = pl.read_chip_biweekly(row["tif"], "sentinel1")
    x = torch.from_numpy(pl.fill_nan(np.ascontiguousarray(
        chip[..., :2].transpose(2, 0, 1))))[None]
    return F.interpolate(x, size=(224, 224), mode="bilinear", align_corners=False)[0]

M2, SD2 = tokb.M["sentinel2"], tokb.SD["sentinel2"]
def gen_input_s2(site, period_id):
    row = idx.query("site_id == @site and sensor == 'sentinel2' and "
                    "period_id == @period_id").iloc[0]
    chip = pl.read_chip_biweekly(row["tif"], "sentinel2")
    x6 = torch.from_numpy(pl.fill_nan(np.ascontiguousarray(
        chip[..., :6].transpose(2, 0, 1))))[None] * 10_000.0
    x6 = F.interpolate(x6, size=(224, 224), mode="bilinear", align_corners=False)[0]
    x12 = torch.zeros(12, 224, 224)
    for j in range(12):
        x12[j] = float(M2[j])          # absent bands at the pretraining mean (raw DN)
    for ci, j in enumerate(tokb.IDX12):
        x12[j] = x6[ci]
    return x12

# sanity: the generator standardizes S2L2A input with the same stats our slots assume
from terratorch.models.backbones.terramind.model.terramind_register import (
    v1_pretraining_mean, v1_pretraining_std)
for key in ("untok_sen2l2a@224",):
    if key in v1_pretraining_mean:
        assert np.allclose(v1_pretraining_mean[key],
                           v1_pretraining_mean["tok_sen2l2a@224"]), key
        assert np.allclose(v1_pretraining_std[key],
                           v1_pretraining_std["tok_sen2l2a@224"]), key
print("S2 pretraining stats consistent (tok vs untok)")

GEN_BATCH = 8
cacheA = LATD / "genfill_s2_chips.npz"
gs2 = {}
if cacheA.exists():
    _z = np.load(cacheA)
    gs2 = {tuple(k.split("|")): _z[k] for k in _z.files}
    print(f"arm A cache loaded: {len(gs2)} chips")

todoA = [r for r in genA.itertuples() if (r.site_id, r.period_id) not in gs2]
print(f"arm A: {len(todoA)} of {len(genA)} to generate")
if todoA:
    genr = pl.load_generator("terramind_v1_large_generate", inp="S1GRD", out="S2L2A")
    for s0 in range(0, len(todoA), GEN_BATCH):
        chunk = todoA[s0:s0 + GEN_BATCH]
        x = torch.stack([gen_input_s1(r.site_id, r.period_id) for r in chunk])
        out = genr.generate(x)                        # [B,12,224,224] DN units
        assert out.shape[1:] == (12, 224, 224), tuple(out.shape)
        if s0 == 0:   # determinism gate: regenerate the first batch
            out2 = genr.generate(x)
            g = float((out - out2).abs().max())
            print(f"generation determinism gate: max|diff| = {g:.2e} DN")
            assert g < 1.0, g
            b8 = out[0, tokb.IDX12[3]] / 10_000.0     # NIR reflectance sanity
            assert 0.0 < float(b8.mean()) < 1.0, float(b8.mean())
        for r, o in zip(chunk, out):
            gs2[(r.site_id, r.period_id)] = \
                o[tokb.IDX12].numpy().astype(np.float16)   # 6 project bands, DN
        if (s0 // GEN_BATCH) % 10 == 0 or s0 + GEN_BATCH >= len(todoA):
            print(f"arm A: {min(s0 + GEN_BATCH, len(todoA))}/{len(todoA)}",
                  flush=True)
    np.savez_compressed(cacheA, **{"|".join(k): v for k, v in gs2.items()})
    print("saved:", cacheA, f"({cacheA.stat().st_size/1e6:.0f} MB)")
assert len(gs2) == len(genA)

## Arm B — generate whole missing S1 images from same-period S2

In [ ]:
cacheB = LATD / "genfill_s1_chips.npz"
gs1 = {}
if cacheB.exists():
    _z = np.load(cacheB)
    gs1 = {tuple(k.split("|")): _z[k] for k in _z.files}
    print(f"arm B cache loaded: {len(gs1)} chips")

todoB = [r for r in genB.itertuples() if (r.site_id, r.period_id) not in gs1]
print(f"arm B: {len(todoB)} of {len(genB)} to generate")
if todoB:
    genr_b = pl.load_generator("terramind_v1_large_generate", inp="S2L2A", out="S1GRD")
    for s0 in range(0, len(todoB), GEN_BATCH):
        chunk = todoB[s0:s0 + GEN_BATCH]
        x = torch.stack([gen_input_s2(r.site_id, r.period_id) for r in chunk])
        out = genr_b.generate(x)                      # [B,2,224,224] dB units
        assert out.shape[1:] == (2, 224, 224), tuple(out.shape)
        if s0 == 0:
            vv = float(out[0, 0].mean())
            assert -35.0 < vv < 5.0, vv               # dB-range sanity
        for r, o in zip(chunk, out):
            gs1[(r.site_id, r.period_id)] = o.numpy().astype(np.float16)
        print(f"arm B: {min(s0 + GEN_BATCH, len(todoB))}/{len(todoB)}", flush=True)
    np.savez_compressed(cacheB, **{"|".join(k): v for k, v in gs1.items()})
    print("saved:", cacheB)
assert len(gs1) == len(genB)

## Encode the genfill panel

Latents: original cache for every untouched image; replaced S2 chips encoded via
`prep_s2_dn224` (DN@224, no second resize); arm-B S1 via `prep_s1_db224` (dB@224).
Determinism gate on a re-encoded batch, then the cache + manifest are written.

In [ ]:
lat_gen = dict(lat_orig)
enc_jobs = ([("sentinel2", r.site_id, r.period_id,
              tokb.prep_s2_dn224(gs2[(r.site_id, r.period_id)].astype(np.float32)))
             for r in genA.itertuples()] +
            [("sentinel1", r.site_id, r.period_id,
              tokb.prep_s1_db224(gs1[(r.site_id, r.period_id)].astype(np.float32)))
             for r in genB.itertuples()])
BATCH = 64
for sensor in SENSORS:
    rows = [j for j in enc_jobs if j[0] == sensor]
    for s0 in range(0, len(rows), BATCH):
        chunk = rows[s0:s0 + BATCH]
        q = tokb.encode_batch(sensor, [j[3] for j in chunk])
        for j, qi in zip(chunk, q):
            lat_gen[(j[1], sensor, j[2])] = qi.numpy()
    if rows:   # determinism gate: re-encode the first batch
        q2 = tokb.encode_batch(sensor, [j[3] for j in rows[:16]])
        g = max(float(np.abs(lat_gen[(j[1], sensor, j[2])] - qi.numpy()).max())
                for j, qi in zip(rows[:16], q2))
        print(f"encode determinism gate {sensor}: max|diff| = {g:.2e}")
        assert g < 1e-5
    print(f"{sensor}: {len(rows)} genfill latents encoded")

n_expected = 2322 + len(genB)
assert len(lat_gen) == n_expected, (len(lat_gen), n_expected)
manifest = {
    "date": "2026-08-20",
    "base": "latents_biweekly.npz (chip-mean run, notebook 01)",
    "generation": {"model": "terramind_v1_large_generate",
                   "recipe": "IBM repo verbatim: pretrained=True, standardize=True, "
                             "timesteps=10, full-image replacement",
                   "armA_s1_to_s2": len(genA), "armB_s2_to_s1": len(genB),
                   "chipmean_fallback": int((prov["fill_method"] ==
                                             "chipmean_fallback").sum()),
                   "seed": pl.SEED},
    "fill_scope": "ALL S2 images incl. treated P09/P10 (encoding only); "
                  "ground truth always raw observed pixels",
    "n_latents": len(lat_gen),
}
pl.save_latents(lat_gen, LATD / "latents_biweekly_genfill.npz", manifest)
print("saved:", LATD / "latents_biweekly_genfill.npz")

idx_gen = idx.copy()
idx_gen["has_latent"] = [
    (r.site_id, r.sensor, r.period_id) in lat_gen for r in idx_gen.itertuples()]
assert int(idx_gen["has_latent"].sum()) == n_expected

In [ ]:
# example strip: observed (clouds as gaps) vs chip-mean fill vs generated, S2 RGB
ex = (genA[genA["site_id"].str.startswith("treatment")]
      .sort_values("valid_pixel_fraction").iloc[0])
ex_site, ex_pid, ex_vf = ex["site_id"], ex["period_id"], ex["valid_pixel_fraction"]
row = idx[(idx["site_id"] == ex_site) & (idx["sensor"] == "sentinel2")
          & (idx["period_id"] == ex_pid)].iloc[0]
chip = pl.read_chip_biweekly(row["tif"], "sentinel2")
def rgb(c3):
    return np.clip(np.stack(c3, axis=-1) / 0.30, 0, 1)
obs = rgb([chip[..., 2], chip[..., 1], chip[..., 0]])
obs[~np.isfinite(chip[..., :3]).all(axis=-1)] = 1.0     # clouds shown white
cm = pl.fill_nan(np.ascontiguousarray(chip[..., :6].transpose(2, 0, 1)))
cmr = rgb([cm[2], cm[1], cm[0]])
g6 = gs2[(ex_site, ex_pid)].astype(np.float32) / 10_000.0
g101 = F.interpolate(torch.from_numpy(g6)[None], size=(101, 101), mode="bilinear",
                     align_corners=False)[0].numpy()
gr = rgb([g101[2], g101[1], g101[0]])
fig, axes = plt.subplots(1, 3, figsize=(9.5, 3.4))
for ax, im, t in zip(axes, [obs, cmr, gr],
                     [f"observed ({ex_vf:.0%} valid, clouds white)",
                      "chip-mean fill (notebook-01 input)",
                      "generated from same-period S1 (this input)"]):
    ax.imshow(im); ax.set_title(t, fontsize=8)
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
fig.suptitle(f"{ex_site} {ex_pid} — S2 true color")
fig.tight_layout()
fig.savefig(TS / "panel_genfill_example.png", bbox_inches="tight")
plt.show()
print("figure saved")

## Pipeline rerun on the genfill panel — kNN → SCM → ASCM

Same modules, main arm only (the quality-40 arm is superseded: there are no low-validity
inputs left to filter). Note the pooled scaler is refit on the genfill latents, so RMSEs
are in this run's own SD units — levels are compared via each run's own 1-SD test, not
raw numbers.

In [ ]:
panel_gen = pl.Panel(lat_gen, idx_gen, roster)
don_g, donors_g, _ = pl.knn_donors(panel_gen, arms=("main",))
don_g.to_csv(TS / "panel_genfill_knn_donors.csv", index=False)
print("saved:", TS / "panel_genfill_knn_donors.csv", don_g.shape)

ov_cov = pl.donor_overlap(don_g, panel_gen)
print("\ndonor overlap with the 5 covariate-matched controls:")
for sensor in SENSORS:
    print(f"  {sensor}: {ov_cov[sensor]}  (mean {np.mean(ov_cov[sensor]):.1f} of 5)")
print("\ndonor stability vs the chip-mean run (overlap of the 5 kNN donors):")
for sensor in SENSORS:
    ov = [len(set(donors_g["main"][(sensor, t)]) &
              set(donors0["main"][(sensor, t)])) for t in panel_gen.treatments]
    print(f"  {sensor}: {ov}  (mean {np.mean(ov):.1f} of 5)")

scm_g = panel_scm.run_scm(panel_gen, donors_g, arms=("main",))
scm_g["weights"].to_csv(TS / "panel_genfill_scm_weights.csv", index=False)
scm_g["validation"].to_csv(TS / "panel_genfill_scm_validation.csv", index=False)
scm_g["effects"].to_csv(TS / "panel_genfill_scm_effects.csv", index=False)
print("saved: panel_genfill_scm_{weights,validation,effects}.csv")

ascm_g = panel_ascm.run_ascm(panel_gen, donors_g, scm_g, arms=("main",))
ascm_g["weights"].to_csv(TS / "panel_genfill_ascm_weights.csv", index=False)
ascm_g["lambda_cv"].to_csv(TS / "panel_genfill_ascm_lambda_cv.csv", index=False)
ascm_g["validation"].to_csv(TS / "panel_genfill_ascm_validation.csv", index=False)
ascm_g["vs_scm"].to_csv(TS / "panel_genfill_scm_vs_ascm.csv", index=False)
ascm_g["effects"].to_csv(TS / "panel_genfill_ascm_effects.csv", index=False)
print("saved: panel_genfill_ascm_{weights,lambda_cv,validation,effects}.csv, "
      "panel_genfill_scm_vs_ascm.csv")

print("\n=== genfill SCM validation (own pooled-SD units) ===")
vg = scm_g["validation"]
print("frozen_joint:")
print(vg.query("scheme == 'frozen_joint'").pivot_table(
    index="treatment_site_id", columns="sensor", values="valid_rmse").round(3))
print("expanding:")
print(vg.query("scheme == 'expanding'").pivot_table(
    index="treatment_site_id", columns=["sensor", "eval_period"],
    values="valid_rmse").round(3))
print("\nn_train_periods (w8) — arm B should have recovered dropped S1 periods:")
w8n = (scm_g["weights"].query("fit_window == 'P01-08'")
       .drop_duplicates(["sensor", "treatment_site_id"]))
print(w8n.pivot_table(index="treatment_site_id", columns="sensor",
                      values="n_train_periods"))

## Decode vs ground truth + effects (both methods)

Reconstruction floor = the genfill treated P09/P10 latents decoded straight back —
the error a *perfect* donor fit could not undercut in this pipeline (it now includes
the generation step for cloudy chips). Truth side: raw observed chips only.

In [ ]:
W_g = {"scm": pl.weights_dict(scm_g["weights"], donors_g),
       "ascm": pl.weights_dict(ascm_g["weights"], donors_g)}
dec_of = pl.decode_all(panel_gen, tokb, donors_g["main"], W_g)
dv_g = pl.decode_validation_table(panel_gen, tokb, donors_g["main"], W_g, dec_of)
dv_g.to_csv(TS / "panel_genfill_decode_validation.csv", index=False)
print("saved:", TS / "panel_genfill_decode_validation.csv", dv_g.shape)

eff_g = pl.effect_features_table(panel_gen, tokb, dec_of)
eff_g.to_csv(TS / "panel_genfill_effect_features.csv", index=False)
print("saved:", TS / "panel_genfill_effect_features.csv", eff_g.shape)

print("\nLATENT space — synthetic vs observed-genfill latent RMSE (own SD units):")
print(dv_g.query("space == 'latent'").groupby(["sensor", "method"])["value"]
      .mean().round(3))
print("\nFEATURE space — mean |error| per band vs raw observed features:")
fa = (dv_g.query("space == 'feature'").assign(a=lambda t: t["value"].abs())
      .pivot_table(index=["sensor", "band"], columns="method", values="a"))
print(fa[["scm", "ascm", "floor"]].round(4))
print("\nmean effect per band across sites x post-periods (SCM):")
print(eff_g.query("method == 'scm'").groupby(["sensor", "band"])["effect"]
      .agg(["mean", "std", "count"]).round(4))

## Comparison with the chip-mean run

Latent-space RMSEs are each in their own run's pooled-SD units (different scalers) —
the honest comparisons are each run's **own 1-SD level test** and the **feature-space
decode errors**, which are in native physical units on the SAME raw observed truth.

In [ ]:
v0 = pd.read_csv(TS / "panel_scm_validation.csv").query("arm == 'main'")
va0 = pd.read_csv(TS / "panel_ascm_validation.csv").query("arm == 'main'")
dv0 = pd.read_csv(TS / "panel_decode_validation.csv")

cmp_rows = []
for scheme_ep in [("frozen_joint", "pooled"), ("expanding", "P09"),
                  ("expanding", "P10")]:
    sch, ep = scheme_ep
    for sensor in SENSORS:
        for tid in panel_gen.treatments:
            def get(df, col="valid_rmse"):
                s = df.query("sensor == @sensor and treatment_site_id == @tid and "
                             "scheme == @sch and eval_period == @ep")[col]
                return float(s.iloc[0]) if len(s) else np.nan
            cmp_rows.append({
                "sensor": sensor, "treatment_site_id": tid, "scheme": sch,
                "eval_period": ep,
                "scm_chipmean": get(v0), "scm_genfill": get(scm_g["validation"]),
                "ascm_chipmean": get(va0), "ascm_genfill": get(ascm_g["validation"]),
                "scm_level_chipmean": get(v0, "flag_level"),
                "scm_level_genfill": get(scm_g["validation"], "flag_level")})
cmp_df = pd.DataFrame(cmp_rows)
cmp_df.to_csv(TS / "panel_genfill_vs_chipmean.csv", index=False)
print("saved:", TS / "panel_genfill_vs_chipmean.csv", cmp_df.shape)

print("\nvalidation RMSE, mean over sites (each run in its OWN SD units):")
print(cmp_df.groupby(["sensor", "scheme", "eval_period"])
      [["scm_chipmean", "scm_genfill", "ascm_chipmean", "ascm_genfill"]]
      .mean().round(3))
print("\nlevel-test passes (valid RMSE < 1 own-SD), of 30 site x scheme cells:")
for col in ("scm_level_chipmean", "scm_level_genfill"):
    print(f"  {col}: {(cmp_df[col] == 'pass').sum()}")

print("\nFEATURE-space decode |error| (native units, same raw truth) — "
      "chip-mean vs genfill:")
f0 = (dv0.query("space == 'feature'").assign(a=lambda t: t["value"].abs())
      .pivot_table(index=["sensor", "band"], columns="method", values="a"))
fg = (dv_g.query("space == 'feature'").assign(a=lambda t: t["value"].abs())
      .pivot_table(index=["sensor", "band"], columns="method", values="a"))
side = pd.concat({"chipmean": f0[["scm", "floor"]],
                  "genfill": fg[["scm", "floor"]]}, axis=1)
print(side.round(4))

In [ ]:
pl.plot_validation_bars(scm_g["validation"], panel_gen,
                        TS / "panel_genfill_scm_validation.png",
                        title_note="genfill SCM")
pl.plot_effect_trajectories(scm_g["traj"], panel_gen,
                            TS / "panel_genfill_scm_effect_trajectories.png",
                            title_note="genfill SCM")
pl.plot_scm_vs_ascm(ascm_g["vs_scm_full"], TS / "panel_genfill_scm_vs_ascm.png")
lam_tab = ascm_g["weights"].drop_duplicates(
    ["arm", "sensor", "treatment_site_id", "fit_window"])
pl.plot_lambda(lam_tab, TS / "panel_genfill_ascm_lambda.png")
pl.plot_decode_features(dv_g, TS / "panel_genfill_decode_validation_features.png")
for sensor, band in (("sentinel2", "NDVI"), ("sentinel1", "VV")):
    pl.plot_site_trajectories(panel_gen, tokb, dec_of, sensor, band,
                              TS / f"panel_genfill_trajectories_{sensor}_{band}.png")
plt.show()

# side-by-side: chip-mean vs genfill S2 validation per site (each in own SD units,
# read against the shared 1-SD line)
fig, ax = plt.subplots(figsize=(7.5, 3.4))
t = cmp_df.query("sensor == 'sentinel2' and scheme == 'frozen_joint'")
x = np.arange(len(t))
ax.bar(x - 0.15, t["scm_chipmean"], 0.3, label="chip-mean fill (notebooks 01-04)")
ax.bar(x + 0.15, t["scm_genfill"], 0.3, label="generation fill (this notebook)")
ax.axhline(1.0, color="k", lw=0.8, ls="--")
ax.text(0.02, 1.02, "1 own-SD", transform=ax.get_yaxis_transform(), fontsize=7)
ax.set_xticks(x)
ax.set_xticklabels([s[-2:] for s in t["treatment_site_id"]])
ax.set_xlabel("treatment site")
ax.set_ylabel("valid RMSE (own SD units)")
ax.set_title("sentinel2 SCM frozen-joint validation — fill method comparison")
ax.legend(frameon=False, fontsize=7)
fig.tight_layout()
fig.savefig(TS / "panel_genfill_comparison.png", bbox_inches="tight")
plt.show()
print("figures saved")

## Reading

*(filled after execution)*